# Project 1: Sales Performance Dashboard (Retail & E-Commerce)
## Exploratory Data Analysis, Data Quality Audit & Star-Schema Pipeline

### Overview & Objectives
This notebook executes a production-grade data engineering pipeline on a retail transactions dataset.
It addresses dirty-data edge cases (nulls, duplicates, negative amounts, multi-format dates), establishes financial reconciliation, and builds an enterprise Star Schema (Fact and Dimension tables) ready for Oracle SQL and Power BI modeling.

### Key Acceptance Criteria:
- Top 10 products and monthly sales trends identified and verified.
- Total net sales reconciled across CSV, Oracle SQL, and Power BI within **0.1% tolerance**.
- Support at least **5 executive KPIs** and **3 multi-dimensional slicers**.
- Rigorous resolution of key test cases: null/duplicate rows, negative amounts, date format errors.

In [1]:
import os
import numpy as np
import pandas as pd
from datetime import datetime

# Paths
RAW_CSV_PATH = '../data/raw/kaggle_retail_sales_raw.csv'
PROCESSED_DIR = '../data/processed/'
os.makedirs(PROCESSED_DIR, exist_ok=True)

print('Environment and libraries ready.')

Environment and libraries ready.


## 1. Data Ingestion & Initial Quality Audit
We inspect the raw CSV dataset and identify nulls, duplicates, date issues, and arithmetic discrepancies.

In [2]:
df_raw = pd.read_csv(RAW_CSV_PATH)
print(f'Raw dataset shape: {df_raw.shape}')
print('\nData Types and Non-Null Counts:')
print(df_raw.info())
df_raw.head(5)

Raw dataset shape: (5055, 17)

Data Types and Non-Null Counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5055 entries, 0 to 5054
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction_ID    5055 non-null   object 
 1   Date              5055 non-null   object 
 2   Customer_ID       5035 non-null   object 
 3   Gender            5040 non-null   object 
 4   Age               5055 non-null   int64  
 5   Product_Category  5040 non-null   object 
 6   Product_ID        5055 non-null   object 
 7   Product_Name      5055 non-null   object 
 8   Quantity          5055 non-null   int64  
 9   Price_Per_Unit    5055 non-null   float64
 10  Discount_Pct      5055 non-null   float64
 11  Total_Amount      5040 non-null   float64
 12  Payment_Method    5055 non-null   object 
 13  Store_ID          5055 non-null   object 
 14  City              5055 non-null   object 
 15  State             5055 non

,Transaction_ID,Date,Customer_ID,Gender,Age,Product_Category,Product_ID,Product_Name,Quantity,Price_Per_Unit,Discount_Pct,Total_Amount,Payment_Method,Store_ID,City,State,Region
0,TXN-000001,2024-11-23,CUST-0229,Male,65,Home & Kitchen,PROD-H302,Non-Stick Ceramic Cookware Set 10pc,1,119.99,0.10,107.99,Credit Card,STR-109,Seattle,WA,West
1,TXN-000002,2024-07-28,CUST-0865,Male,19,Electronics,PROD-E104,Mechanical Gaming Keyboard,1,89.99,0.05,85.49,Cash,STR-104,Dallas,TX,Central
2,TXN-000003,2023-06-13,CUST-0860,Male,46,Sports & Outdoors,PROD-S503,Insulated Stainless Steel Water Bottle,3,19.99,0.00,59.97,UPI / Cash App,STR-107,Los Angeles,CA,West
3,TXN-000004,2023-12-19,CUST-0570,Male,31,Home & Kitchen,PROD-H301,Programmable Espresso Coffee Maker,1,129.99,0.00,129.99,UPI / Cash App,STR-110,Denver,CO,West
4,TXN-000005,2024-07-19,CUST-0089,Other,47,Sports & Outdoors,PROD-S501,Anti-Slip High-Density Yoga Mat,8,29.99,0.00,239.92,Cash,STR-105,Atlanta,GA,South


In [3]:
# Key Test Case 1: Detect Duplicates
exact_dups = df_raw.duplicated().sum()
key_dups = df_raw.duplicated(subset=['Transaction_ID']).sum()
print(f'Exact Duplicate Rows: {exact_dups}')
print(f'Duplicate Transaction IDs: {key_dups}')

# Key Test Case 2: Missing / Null Values
print('\nMissing values count per column:')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

# Key Test Case 3: Negative Quantities and Amounts
neg_qty = (df_raw['Quantity'] <= 0).sum()
neg_amt = (df_raw['Total_Amount'] <= 0).sum()
print(f'\nNegative / Zero Quantities: {neg_qty}')
print(f'Negative / Zero Total Amounts: {neg_amt}')

Exact Duplicate Rows: 40
Duplicate Transaction IDs: 55

Missing values count per column:
Customer_ID         20
Gender              15
Product_Category    15
Total_Amount        15
dtype: int64

Negative / Zero Quantities: 25
Negative / Zero Total Amounts: 50


## 2. Multi-Format Date Parsing & Quality Filtering
The raw dataset contains mixed date formats (`YYYY-MM-DD`, `DD/MM/YYYY`, `MM-DD-YYYY`) and corrupt entries (`2024-02-31`, blank spaces).

In [4]:
def parse_multi_format_date(val):
    if pd.isna(val):
        return pd.NaT
    s = str(val).strip()
    if not s or s.lower() in ('nan', 'none', 'null'):
        return pd.NaT
    formats = ['%Y-%m-%d', '%d/%m/%Y', '%m-%d-%Y', '%Y/%m/%d', '%d-%m-%Y', '%m/%d/%Y']
    for fmt in formats:
        try:
            p = datetime.strptime(s, fmt)
            if 2020 <= p.year <= 2026:
                return pd.Timestamp(p)
        except (ValueError, TypeError):
            continue
    return pd.NaT

df_raw['Parsed_Date'] = df_raw['Date'].apply(parse_multi_format_date)
corrupt_dates = df_raw['Parsed_Date'].isna().sum()
print(f'Corrupt or unparseable dates identified: {corrupt_dates}')

Corrupt or unparseable dates identified: 23


## 3. Data Cleaning Pipeline
- Deduplication
- Text normalization & whitespace trimming
- Imputation of missing customer & category attributes
- Filtering invalid negative sales records
- Arithmetic recalculation: Gross Sales, Discounts, Net Sales, Cost, Gross Profit

In [5]:
# 1. Deduplicate
df_clean = df_raw.drop_duplicates().copy()
df_clean = df_clean.drop_duplicates(subset=['Transaction_ID'], keep='first').copy()

# 2. Drop corrupt dates
df_clean = df_clean.dropna(subset=['Parsed_Date']).copy()

# 3. Filter out negative amounts and quantities
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Total_Amount'] > 0)].copy()

# 4. Text standardization
df_clean['Product_Category'] = df_clean['Product_Category'].fillna('Uncategorized').astype(str).str.strip().str.title()
df_clean['Region'] = df_clean['Region'].astype(str).str.strip().str.title()
df_clean['Customer_ID'] = df_clean['Customer_ID'].fillna('CUST-UNKNOWN').astype(str).str.strip()
df_clean['Gender'] = df_clean['Gender'].fillna('Unknown').astype(str).str.strip()

# 5. Financial recalculations
df_clean['Quantity'] = df_clean['Quantity'].astype(int)
df_clean['Price_Per_Unit'] = df_clean['Price_Per_Unit'].astype(float).round(2)
df_clean['Discount_Pct'] = df_clean['Discount_Pct'].fillna(0.0).astype(float).clip(0.0, 0.90)
df_clean['Gross_Sales'] = (df_clean['Quantity'] * df_clean['Price_Per_Unit']).round(2)
df_clean['Discount_Amount'] = (df_clean['Gross_Sales'] * df_clean['Discount_Pct']).round(2)
df_clean['Net_Sales'] = (df_clean['Gross_Sales'] - df_clean['Discount_Amount']).round(2)

# COGS and Profit
df_clean['Unit_Cost'] = (df_clean['Price_Per_Unit'] * 0.65).round(2)
df_clean['Total_Cost'] = (df_clean['Quantity'] * df_clean['Unit_Cost']).round(2)
df_clean['Gross_Profit'] = (df_clean['Net_Sales'] - df_clean['Total_Cost']).round(2)
df_clean['Profit_Margin_Pct'] = ((df_clean['Gross_Profit'] / df_clean['Net_Sales']) * 100).round(2)
df_clean['Date_Key'] = df_clean['Parsed_Date'].dt.strftime('%Y%m%d').astype(int)

print(f'Cleaned dataset rows: {len(df_clean)}')
print(f'Total Net Sales: ${df_clean["Net_Sales"].sum():,.2f}')

Cleaned dataset rows: 4912
Total Net Sales: $1,071,880.61


## 4. Star-Schema Dimensional Modeling
We normalize the dataset into a high-performance Kimball Star Schema:
- `Dim_Date`
- `Dim_Product`
- `Dim_Customer`
- `Dim_Region`
- `Fact_Sales`

In [6]:
# Dim_Product
dim_product = df_clean[['Product_ID', 'Product_Name', 'Product_Category', 'Price_Per_Unit', 'Unit_Cost']].drop_duplicates(subset=['Product_ID']).reset_index(drop=True)
dim_product['Product_Key'] = dim_product.index + 1

# Dim_Customer
dim_customer = df_clean[['Customer_ID', 'Gender', 'Age']].drop_duplicates(subset=['Customer_ID']).reset_index(drop=True)
dim_customer['Customer_Key'] = dim_customer.index + 1

# Dim_Region
dim_region = df_clean[['Store_ID', 'City', 'State', 'Region']].drop_duplicates(subset=['Store_ID']).reset_index(drop=True)
dim_region['Region_Key'] = dim_region.index + 1

# Fact_Sales
fact_sales = df_clean.merge(dim_product[['Product_Key', 'Product_ID']], on='Product_ID', how='left')
fact_sales = fact_sales.merge(dim_customer[['Customer_Key', 'Customer_ID']], on='Customer_ID', how='left')
fact_sales = fact_sales.merge(dim_region[['Region_Key', 'Store_ID']], on='Store_ID', how='left')
fact_sales['Sales_Key'] = fact_sales.index + 1

print('Dimensional model successfully generated.')
print(f'Dim_Product: {len(dim_product)} rows')
print(f'Dim_Customer: {len(dim_customer)} rows')
print(f'Dim_Region: {len(dim_region)} rows')
print(f'Fact_Sales: {len(fact_sales)} rows')

Dimensional model successfully generated.
Dim_Product: 35 rows
Dim_Customer: 1181 rows
Dim_Region: 10 rows
Fact_Sales: 4912 rows


## 5. Top 10 Products & Monthly Trend Analysis

In [7]:
# Top 10 Products by Net Revenue
top10 = df_clean.groupby('Product_Name')['Net_Sales'].sum().sort_values(ascending=False).head(10)
print('TOP 10 PRODUCTS BY NET SALES:')
display(top10.to_frame('Total_Net_Sales_USD'))

# Monthly Trend
df_clean['Year_Month'] = df_clean['Parsed_Date'].dt.to_period('M').astype(str)
monthly = df_clean.groupby('Year_Month').agg(
    Monthly_Revenue=('Net_Sales', 'sum'),
    Total_Orders=('Transaction_ID', 'count'),
    Units_Sold=('Quantity', 'sum')
)
print('\nMONTHLY SALES TREND:')
display(monthly.head(12))

TOP 10 PRODUCTS BY NET SALES:


,Total_Net_Sales_USD
Product_Name,
Smartphone 5G Pro,138518.21
4K Ultra HD Smart TV 55-inch,102222.89
Robot Vacuum Cleaner Smart Mapping,90836.86
Tablet 10-inch 128GB,88387.73
Camping 4-Person Waterproof Tent,69371.49
Smartwatch Fitness Tracker,47647.52
Adjustable Cast Iron Dumbbell Set,45452.08
Programmable Espresso Coffee Maker,42630.12
Waterproof Winter Down Jacket,40992.15



MONTHLY SALES TREND:


,Monthly_Revenue,Total_Orders,Units_Sold
Year_Month,,,
2023-01,45142.05,204,496
2023-02,49033.85,211,456
2023-03,45541.15,195,478
2023-04,46114.26,217,514
2023-05,42976.36,213,508
2023-06,48718.36,200,466
2023-07,41764.87,187,419
2023-08,39873.99,192,428
2023-09,33709.64,188,379


## 6. Executive KPI Dashboard Summary & Reconciliation

In [8]:
total_sales = df_clean['Net_Sales'].sum()
total_orders = len(df_clean)
units_sold = df_clean['Quantity'].sum()
aov = total_sales / total_orders
total_profit = df_clean['Gross_Profit'].sum()
profit_margin = (total_profit / total_sales) * 100

print('====================================================')
print('       EXECUTIVE KPI SUMMARY SCORECARD              ')
print('====================================================')
print(f'1. Total Net Sales        : ${total_sales:,.2f}')
print(f'2. Total Orders (Volume)  : {total_orders:,}')
print(f'3. Total Units Sold       : {units_sold:,}')
print(f'4. Average Order Value    : ${aov:,.2f}')
print(f'5. Gross Profit Margin    : {profit_margin:.2f}% (${total_profit:,.2f})')
print('====================================================')

       EXECUTIVE KPI SUMMARY SCORECARD              
1. Total Net Sales        : $1,071,880.61
2. Total Orders (Volume)  : 4,912
3. Total Units Sold       : 11,125
4. Average Order Value    : $218.22
5. Gross Profit Margin    : 30.74% ($329,538.23)
